In [1]:
# Pure ResNet18 baseline -- no augmentation, no custom head, no freezing.
# Train on 0° only, test on 0/90/180/270 to measure rotational invariance.

import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

DATA_DIR = r"E:\Deep Learning Spring 26\dataset"
BATCH_SIZE = 32
EPOCHS = 6
LR = 1e-4
NUM_WORKERS = 0
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
# load data and split 80/10/10
train_dir = os.path.join(DATA_DIR, "train")
df = pd.read_csv(os.path.join(DATA_DIR, "train_labels.csv"))

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df.label, random_state=SEED)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

Train : 176020
Val   : 22002
Test  : 22003


In [3]:
# just normalize, no augmentation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CancerDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, rotation=0):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.rotation = rotation

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.img_dir, row.id + ".tif")
        img = Image.open(path).convert("RGB")
        if self.rotation != 0:
            img = img.rotate(self.rotation)
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row.label, dtype=torch.float32)
        return img, label

In [4]:
# data loaders
train_loader = DataLoader(
    CancerDataset(train_df, train_dir, transform, rotation=0),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

val_loader = DataLoader(
    CancerDataset(val_df, train_dir, transform, rotation=0),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

test_loaders = {}
for angle in [0, 90, 180, 270]:
    test_loaders[angle] = DataLoader(
        CancerDataset(test_df, train_dir, transform, rotation=angle),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [5]:
# pure ResNet18 -- just swap the final fc layer to output 1 logit
model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [6]:
def train_one_epoch():
    model.train()
    total_loss = 0
    preds_list, labels_list = [], []

    for imgs, labels in tqdm(train_loader, desc="Training", leave=False):
        imgs = imgs.to(device)
        labels = labels.unsqueeze(1).to(device)

        out = model(imgs)
        loss = criterion(out, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        probs = torch.sigmoid(out).squeeze().detach().cpu().numpy()
        preds_list.extend(probs)
        labels_list.extend(labels.squeeze().cpu().numpy())

    avg_loss = total_loss / len(train_loader)
    preds_arr = np.array(preds_list)
    labels_arr = np.array(labels_list).astype(int)
    binary = (preds_arr >= 0.5).astype(int)

    return (avg_loss, accuracy_score(labels_arr, binary),
            f1_score(labels_arr, binary, average="weighted"),
            roc_auc_score(labels_arr, preds_arr))


def evaluate(loader):
    model.eval()
    preds_list, labels_list = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            probs = torch.sigmoid(model(imgs)).squeeze().cpu().numpy()
            preds_list.extend(probs)
            labels_list.extend(labels.numpy())

    preds_arr = np.array(preds_list)
    labels_arr = np.array(labels_list).astype(int)
    binary = (preds_arr >= 0.5).astype(int)

    return (accuracy_score(labels_arr, binary),
            f1_score(labels_arr, binary, average="weighted"),
            roc_auc_score(labels_arr, preds_arr))

In [7]:
# training loop
best_auc = 0.0

for epoch in range(1, EPOCHS + 1):
    loss, train_acc, train_f1, train_auc = train_one_epoch()
    scheduler.step()
    val_acc, val_f1, val_auc = evaluate(val_loader)

    lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch}/{EPOCHS} (lr={lr:.2e})")
    print(f"  Train  =>  Loss: {loss:.4f}  Acc: {train_acc:.4f}  F1: {train_f1:.4f}  AUC: {train_auc:.4f}")
    print(f"  Val    =>  Acc: {val_acc:.4f}  F1: {val_f1:.4f}  AUC: {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_model_baseline.pth")
        print(f"  >> Saved best model (AUC={val_auc:.4f})")


  EPOCH 1/6  (LR = 9.33e-05)
  Train                | Loss=0.1997 | Acc=0.9231 | F1=0.9228 | AUC=0.9731
  Validation           | Acc=0.9538 | F1=0.9537 | AUC=0.9879
  ** New best model saved (Val AUC=0.9879) **



  EPOCH 2/6  (LR = 7.50e-05)
  Train                | Loss=0.1174 | Acc=0.9575 | F1=0.9575 | AUC=0.9901
  Validation           | Acc=0.9600 | F1=0.9599 | AUC=0.9911
  ** New best model saved (Val AUC=0.9911) **



  EPOCH 3/6  (LR = 5.00e-05)
  Train                | Loss=0.0710 | Acc=0.9748 | F1=0.9748 | AUC=0.9962
  Validation           | Acc=0.9616 | F1=0.9615 | AUC=0.9913
  ** New best model saved (Val AUC=0.9913) **



  EPOCH 4/6  (LR = 2.50e-05)
  Train                | Loss=0.0357 | Acc=0.9879 | F1=0.9879 | AUC=0.9990
  Validation           | Acc=0.9637 | F1=0.9637 | AUC=0.9930
  ** New best model saved (Val AUC=0.9930) **



  EPOCH 5/6  (LR = 6.70e-06)
  Train                | Loss=0.0144 | Acc=0.9953 | F1=0.9953 | AUC=0.9998
  Validation           | Acc=0.9686 | F1=0.9686 | AUC=0.9934
  ** New best model saved (Val AUC=0.9934) **



  EPOCH 6/6  (LR = 0.00e+00)
  Train                | Loss=0.0058 | Acc=0.9983 | F1=0.9983 | AUC=1.0000
  Validation           | Acc=0.9702 | F1=0.9702 | AUC=0.9941
  ** New best model saved (Val AUC=0.9941) **

  ROTATION SYMMETRY TEST (using best model)



In [8]:
# rotation test
print("\nRotation test (best model):")
model.load_state_dict(torch.load("best_model_baseline.pth"))

for angle in [0, 90, 180, 270]:
    acc, f1, auc = evaluate(test_loaders[angle])
    print(f"  {angle:3d}°  =>  Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")

C:\Users\AhmedFahmy\AppData\Local\Temp\ipykernel_22772\3088562970.py:319: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model_baseline

  Test @   0°          | Acc=0.9699 | F1=0.9699 | AUC=0.9942
  Test @  90°          | Acc=0.9407 | F1=0.9404 | AUC=0.9798
  Test @ 180°          | Acc=0.9450 | F1=0.9447 | AUC=0.9830
  Test @ 270°          | Acc=0.9418 | F1=0.9415 | AUC=0.9814

  Experiment complete!
